<a href="https://colab.research.google.com/github/dhruvjoshi0905/Hack-O-Week/blob/main/hack_o_week(14)ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install cryptography pandas

In [10]:
import json
import time
import random
import pandas as pd
from datetime import datetime
from cryptography.fernet import Fernet

# ==========================================
# 1. KEY MANAGEMENT (The Zero-Knowledge Rule)
# ==========================================
# In E2E, the database NEVER gets this key. It only lives on the smartwatch and the doctor's dashboard.
e2e_secret_key = Fernet.generate_key()
cipher = Fernet(e2e_secret_key)

# Our simulated Cloud Database (Managed by Affan & Aaditi)
cloud_database = []

In [12]:
# ==========================================
# 2. THE WEARABLE DEVICE (Client-Side Encryption)
# ==========================================
def generate_and_encrypt_payload(device_id="watch_X"):
    """
    Simulates the watch generating JSON data and encrypting it locally.
    """
    # 1. Generate Raw JSON Data
    raw_telemetry = {
        "device_id": device_id,
        "timestamp": datetime.now().isoformat(),
        "heart_rate_bpm": random.randint(65, 115),
        "blood_oxygen_spo2": random.randint(95, 100)
    }

    # 2. Serialize to string, then encode to bytes
    json_string = json.dumps(raw_telemetry)
    byte_payload = json_string.encode('utf-8')

    # 3. Encrypt the payload ON THE DEVICE
    encrypted_payload = cipher.encrypt(byte_payload)

    print(f"[Smartwatch] Generated Data: {json_string}")
    print(f"[Smartwatch] Scrambled Ciphertext: {encrypted_payload[:40]}...\n")

    return encrypted_payload

In [13]:
# ==========================================
# 3. THE BACKEND SERVER (Cloud Storage)
# ==========================================
def backend_store_data(ciphertext):
    """
    The server acts purely as a storage locker. It cannot read the payload.
    """
    record = {
        "record_id": len(cloud_database) + 1,
        "secure_payload": ciphertext,
        "stored_at": datetime.now().strftime("%H:%M:%S")
    }
    cloud_database.append(record)
    print(f"[Backend Cloud] Safely stored encrypted record #{record['record_id']}.\n")

In [14]:
# ==========================================
# 4. THE AUTHORIZED DASHBOARD (Client-Side Decryption)
# ==========================================
def fetch_and_decrypt_dashboard(record_id):
    """
    The dashboard downloads the scrambled data and unlocks it using the secret key.
    """
    # Fetch the ciphertext from the database
    record = next((item for item in cloud_database if item["record_id"] == record_id), None)

    if record:
        try:
            # Decrypt the payload back to bytes
            decrypted_bytes = cipher.decrypt(record["secure_payload"])

            # Convert back to a usable JSON dictionary
            decrypted_json = json.loads(decrypted_bytes.decode('utf-8'))

            print(f"[Doctor Dashboard] Decrypted Record #{record_id} Successfully:")
            print(json.dumps(decrypted_json, indent=2))
        except Exception as e:
            print(f"[Dashboard] Decryption failed. Error: {e}")

In [15]:
# ==========================================
# 5. EXECUTE THE SIMULATION
# ==========================================
print("--- Phase 1: Live Data Ingestion & Encryption ---")
# Generate 3 bursts of wearable data
for _ in range(3):
    secure_packet = generate_and_encrypt_payload()
    backend_store_data(secure_packet)
    time.sleep(0.5)

print("\n--- Phase 2: Database Audit (Proving Zero-Knowledge) ---")
# Show the faculty exactly what the database looks like.
# It proves that even if the database is hacked, no health data is exposed.
db_dataframe = pd.DataFrame(cloud_database)
print("If a hacker breaches the database, this is all they see:")
print(db_dataframe[['record_id', 'secure_payload']])

print("\n--- Phase 3: Authorized Access ---")
# Simulate an authorized user viewing the second record
fetch_and_decrypt_dashboard(record_id=2)

--- Phase 1: Live Data Ingestion & Encryption ---
[Smartwatch] Generated Data: {"device_id": "watch_X", "timestamp": "2026-04-01T17:51:27.845139", "heart_rate_bpm": 98, "blood_oxygen_spo2": 100}
[Smartwatch] Scrambled Ciphertext: b'gAAAAABpzVsf_NSjscT64fuY0wgkRuxUHhRXIzdE'...

[Backend Cloud] Safely stored encrypted record #1.

[Smartwatch] Generated Data: {"device_id": "watch_X", "timestamp": "2026-04-01T17:51:28.345678", "heart_rate_bpm": 100, "blood_oxygen_spo2": 98}
[Smartwatch] Scrambled Ciphertext: b'gAAAAABpzVsgftAl4xwNlHeJ-z8lBhOkkbqE1wId'...

[Backend Cloud] Safely stored encrypted record #2.

[Smartwatch] Generated Data: {"device_id": "watch_X", "timestamp": "2026-04-01T17:51:28.846330", "heart_rate_bpm": 86, "blood_oxygen_spo2": 95}
[Smartwatch] Scrambled Ciphertext: b'gAAAAABpzVsgoOLe5nLsZxmKXNl_Uus_mKKsVxJi'...

[Backend Cloud] Safely stored encrypted record #3.


--- Phase 2: Database Audit (Proving Zero-Knowledge) ---
If a hacker breaches the database, this is all they s